In [1]:
import pandas as pd
import os
import glob

In [2]:
# Đường dẫn tới folder chứa data đã làm sạch (so với thư mục data/)
data_dir = 'processed/cleaned'

# Lấy tất cả các file csv trong thư mục
all_files = glob.glob(os.path.join(data_dir, '*.csv'))

dfs = []
for f in all_files:
    df = pd.read_csv(f)
    dfs.append(df)

if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    print("Tổng số dòng ban đầu:", len(combined_df))
    
    # Lọc ra dataframe chứa các rows có giá trị cột 'is_noise' = True
    if 'is_noise' in combined_df.columns:
        noise_df = combined_df[combined_df['is_noise'] == True]
        print("\nTổng số dòng có is_noise = True:", len(noise_df))
        display(noise_df)
    else:
        print("\nKhông tìm thấy cột 'is_noise' trong dataset.")
else:
    print('Không tìm thấy dataset nào trong folder', data_dir)


Tổng số dòng ban đầu: 3414

Tổng số dòng có is_noise = True: 15


,id,url,title,lead_paragraph,category,publish_datetime,source,thumbnail_url,label,province_fixed,province_names,is_noise,noise_reasons
130,article_0790,https://baomoi.com/trao-huy-hieu-80-nam-tuoi-d...,Trao Huy hiệu 80 năm tuổi Đảng tặng đảng viên ...,Trao Huy hiệu 80 năm tuổi Đảng tặng đảng viên ...,Tin tức tổng hợp,2025-06-23T06:27:00.000Z,Báo Mới,data/images/article_0790_image.png,non-clickbait,False,NaN,True,title_equals_lead
203,article_0145,https://kenh14.vn/lo-mat-moc-va-body-truoc-khi...,Lộ mặt mộc và body trước khi giảm 10kg của tân...,Phải wow luôn đó!,Sức khỏe & Đời sống,2025-06-23T07:00:00,Kênh14,data/images/article_0145_image.png,clickbait,False,NaN,True,lead_too_short
239,article_2870,https://thanhnien.vn/the-thao/bong-da-thanh-ni...,Top đội bóng,NaN,Chính trị & An ninh,NaN,Thanh Niên,data/images/article_2870_image.png,clickbait,False,NaN,True,lead_too_short
1045,article_1821,https://camnangtuyensinh.thanhnien.vn,Thời sự,NaN,Other,"Thứ Ba, 11/4/2023",Thanh Niên,data/images/article_1821_image.png,clickbait,False,NaN,True,title_too_short|lead_too_short
1243,article_1067,https://baomoi.com/gin-giu-phat-huy-gia-tri-va...,"Gìn giữ, phát huy giá trị văn hóa truyền thống...","Gìn giữ, phát huy giá trị văn hóa truyền thống...",Văn hóa & Nghệ thuật,2025-06-23T06:33:00.000Z,Báo Mới,data/images/article_1067_image.png,non-clickbait,False,NaN,True,title_equals_lead
1592,article_2456,https://baomoi.com/quy-tin-dung-nhan-dan-xa-do...,"Quỹ tín dụng nhân dân xã Đồng Văn, huyện Yên L...","Quỹ tín dụng nhân dân xã Đồng Văn, huyện Yên L...",Chính trị & An ninh,2025-06-23T09:04:52.000Z,Báo Mới,data/images/article_2456_image.png,non-clickbait,False,NaN,True,title_equals_lead
1695,article_1196,https://thanhnien.vn/video/doi-song.htm,Đời sống,"Video phóng sự điều tra các vụ án hình sự, tha...",Chính trị & An ninh,NaN,Thanh Niên,data/images/article_1196_image.png,clickbait,False,NaN,True,title_too_short
1751,article_2646,https://www.saostar.vn/cong-nghe/california-fi...,California Fitness ra mắt Cali AI PT - Trợ lý ...,California Fitness ra mắt Cali AI PT - Trợ lý ...,Công nghệ & Khoa học,2025-06-19T08:00:28.796+07:00,SaoStar,data/images/article_2646_image.png,non-clickbait,False,NaN,True,title_equals_lead
1900,article_2299,https://thanhnien.vn/nho-truong-sa-tho-cua-hoa...,Nhớ Trường Sa - Thơ của Hoàng Thụy Anh,NaN,Văn hóa & Nghệ thuật,2025-06-22T08:00:00+07:00,Thanh Niên,data/images/article_2299_image.png,non-clickbait,False,NaN,True,lead_too_short
1936,article_1931,https://baomoi.com/160-tay-vot-tranh-tai-tai-g...,160 tay vợt tranh tài tại Giải bóng bàn các nh...,160 tay vợt tranh tài tại Giải bóng bàn các nh...,Thể thao,2025-06-23T07:41:00.000Z,Báo Mới,data/images/article_1931_image.png,non-clickbait,False,NaN,True,title_equals_lead


In [3]:
# 1. Loại bỏ các rows có giá trị 'is_noise' = True
cleaned_df = combined_df[combined_df['is_noise'] != True].copy()

In [4]:
# 2. Loại bỏ các rows mà 'title' có chứa đoạn text đặc biệt
texts_to_remove = ['Một bệnh viện cầu cứu khi bị hacker đe dọa', 'Xuân Trường mừng cho Tuấn Anh']
for text in texts_to_remove:
    cleaned_df = cleaned_df[~cleaned_df['title'].str.contains(text, na=False)]

In [ ]:
def clean_title(title):
    if not isinstance(title, str):
        return title
    
    last_dash_idx = title.rfind('-')
    if last_dash_idx != -1:
        suffix = title[last_dash_idx + 1:]
        
        # 1. Đưa suffix về chữ thường để loại bỏ sự khác biệt hoa/thường
        suffix_lower = suffix.lower()
        
        # 2. Định nghĩa keywords hoàn toàn bằng chữ thường
        keywords = ['báo', 'tạp chí', 'bnews', 'chuyên trang', 'đài']
        
        # 3. Kiểm tra trên chuỗi đã viết thường
        if any(keyword in suffix_lower for keyword in keywords):
            title = title[:last_dash_idx]
            
    return title.strip()

cleaned_df['title'] = cleaned_df['title'].apply(clean_title)

In [6]:
pd.set_option('display.max_colwidth', None)

In [31]:
cleaned_df[cleaned_df['title'].str.contains('-', na=False)].sample(5)['title']

1602             Đường Nguyễn Văn Bộ - Nơi ghi dấu và tiếp lửa truyền thống
1162                                                      Tin tức - Sự kiện
3392       Alexandr Wang - thiên tài AI khiến Zuckerberg đặt cược 14 tỷ USD
924     Trụ sở P.Xuân Hương - Đà Lạt rộn ràng trước giờ vận hành thử nghiệm
732                 Lịch cúp điện ở Bà Rịa - Vũng Tàu, Bình Dương ngày 23.6
Name: title, dtype: object

In [53]:
def show_short_titles(text_column, max_text):
    """
    Hiển thị ngẫu nhiên 5 dòng từ cột dữ liệu có số lượng ký tự 
    (không tính khoảng trắng) nhỏ hơn hoặc bằng max_text.
    """
    # 1. Loại bỏ các giá trị NaN/Null để tránh lỗi khi xử lý chuỗi
    valid_texts = text_column.dropna().astype(str)
    
    # 2. Xóa tất cả các khoảng trắng (\s+) và đếm độ dài ký tự còn lại
    # r'\s+' đại diện cho mọi loại khoảng trắng (dấu cách, tab, enter...)
    char_counts = valid_texts.str.replace(r'\s+', '', regex=True).str.len()
    
    # 3. Lọc ra các dòng thỏa mãn điều kiện <= max_text
    filtered_texts = valid_texts[char_counts <= max_text]
    
    # 4. Kiểm tra số lượng kết quả và lấy mẫu
    n_samples = min(5, len(filtered_texts))
    
    if n_samples == 0:
        print(f"Không tìm thấy tiêu đề nào có <= {max_text} ký tự (không tính khoảng trắng).")
        return
        
    # Lấy ngẫu nhiên n_samples
    random_samples = filtered_texts.sample(n=n_samples)
    
    # 5. Hiển thị kết quả trực quan
    print(f"Hiển thị {n_samples} mẫu có độ dài <= {max_text} ký tự (chỉ tính chữ/số):\n")
    for i, title in enumerate(random_samples, 1):
        # Tính lại độ dài để hiển thị đối chứng
        actual_length = len(title.replace(" ", "")) # Cách đếm đơn giản minh họa khi in
        print(f"Mẫu {i} [{actual_length} ký tự]:\n{title}\n{'-'*40}")

In [58]:
# --- CÁCH SỬ DỤNG ---
show_short_titles(cleaned_df['title'], max_text=10)

Hiển thị 3 mẫu có độ dài <= 10 ký tự (chỉ tính chữ/số):

Mẫu 1 [10 ký tự]:
Truyền hình
----------------------------------------
Mẫu 2 [10 ký tự]:
Yêu nhau lắm
----------------------------------------
Mẫu 3 [9 ký tự]:
Đời nghệ sĩ
----------------------------------------


In [63]:
cleaned_df[cleaned_df['title'] == 'Tin tức - Sự kiện']

,id,url,title,lead_paragraph,category,publish_datetime,source,thumbnail_url,label,province_fixed,province_names,is_noise,noise_reasons
1162,article_2063,https://thanhnien.vn/du-lich/tin-tuc-su-kien.htm,Tin tức - Sự kiện,Tin tức dụ lịch - Các sự kiện du lịch tại các tỉnh thành ở Việt Nam và thế giới. Lễ hội du lịch hàng năm tổ chức tại các địa điểm du lịch nổi tiếng.,Chính trị & An ninh,NaN,Thanh Niên,data/images/article_2063_image.png,clickbait,False,NaN,False,NaN


In [64]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3395 entries, 0 to 3413
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                3395 non-null   object
 1   url               3395 non-null   object
 2   title             3395 non-null   object
 3   lead_paragraph    3395 non-null   object
 4   category          3395 non-null   object
 5   publish_datetime  3368 non-null   object
 6   source            3395 non-null   object
 7   thumbnail_url     3391 non-null   object
 8   label             3395 non-null   object
 9   province_fixed    3395 non-null   bool  
 10  province_names    108 non-null    object
 11  is_noise          3395 non-null   bool  
 12  noise_reasons     0 non-null      object
dtypes: bool(2), object(11)
memory usage: 324.9+ KB


In [65]:
# 4. Lưu thành 'Cleaned_Clickbait_Dataset.csv'
import os
output_path = os.path.join(data_dir, 'Cleaned_Clickbait_Dataset.csv')
cleaned_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Đã lưu file làm sạch tại: {output_path}")
print(f"Tổng số dòng sau khi làm sạch: {len(cleaned_df)}")

Đã lưu file làm sạch tại: processed/cleaned/Cleaned_Clickbait_Dataset.csv
Tổng số dòng sau khi làm sạch: 3395
